# Benchmark de conjuntos dominantes mínimos

Use o notebook na raiz do projeto ou em `notebooks/`. Os módulos
`demds.py`, `iterated_greedy.py` e `greedy.py` ficam em
`src/complex_networks/algorithms/mds/`.

Execute **Preparação** e escolha a seção **Análise individual** ou
**Relatórios em lote**. A última célula inicia o lote completo.

## Preparação

In [ ]:
import csv
import gc
import sys
from datetime import datetime
from fractions import Fraction
from math import ceil, sqrt
from pathlib import Path
from time import perf_counter_ns
from uuid import uuid4

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.transforms import offset_copy

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd()))

from src.complex_networks.config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR

IG_BETA = 0.2
otimos_mds = (
    pd.read_csv(
        RESULTS_DIR / "otimos.csv",
        index_col="instance",
        dtype={"instance": str, "dms": "Int64"},
        keep_default_na=False,
        na_values={"dms": ["", "-", "–", "—"]},
    )["dms"]
    .dropna()
    .to_dict()
)

In [ ]:
def medir_execucao(G, instancia, algoritmo, seed, limite_s, solver, argumentos):
    import networkx as nx

    run_id = uuid4().hex
    eventos = []

    def registrar(cardinalidade):
        eventos.append((perf_counter_ns() - inicio_ns, cardinalidade))

    solucao = None
    erro = None
    inicio_ns = perf_counter_ns()
    try:
        solucao = solver(
            G,
            seed=seed,
            on_improvement=registrar,
            **argumentos,
        )
    except Exception as exc:
        fim_ns = perf_counter_ns()
        erro = f"{type(exc).__name__}: {exc}"
    else:
        fim_ns = perf_counter_ns()

    cardinalidade = len(solucao) if solucao is not None else None
    valida = (
        solucao is not None
        and all(v in G for v in solucao)
        and nx.is_dominating_set(G, solucao)
    )
    if erro is None and not valida:
        erro = "O conjunto retornado não domina o grafo."
    if erro is None and (not eventos or eventos[-1][1] != cardinalidade):
        erro = "A última cardinalidade registrada difere da solução retornada."

    resumo = {
        "run_id": run_id,
        "instancia": instancia,
        "algoritmo": algoritmo,
        "seed": seed,
        "n_nodes": G.number_of_nodes(),
        "n_edges": G.number_of_edges(),
        "limite_s": limite_s,
        "relogio_limite": "cpu" if algoritmo == "DemDS" else "decorrido",
        "beta": argumentos.get("beta"),
        "tempo_total_s": (fim_ns - inicio_ns) / 1_000_000_000,
        "cardinalidade_final": cardinalidade,
        "tempo_ate_melhor_s": eventos[-1][0] / 1_000_000_000 if erro is None else None,
        "n_eventos": len(eventos),
        "valida": valida,
        "status": "ok" if erro is None else "erro",
        "erro": erro,
    }
    historico = [
        {
            "run_id": run_id,
            "tempo_s": tempo_ns / 1_000_000_000,
            "cardinalidade": tamanho,
        }
        for tempo_ns, tamanho in eventos
    ]
    return resumo, historico


def salvar_linhas(path, linhas):
    if not linhas:
        return
    precisa_cabecalho = not path.exists()
    with path.open("a", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(linhas[0]))
        if precisa_cabecalho:
            writer.writeheader()
        writer.writerows(linhas)



def criar_pasta_resultados(base):
    base.mkdir(parents=True, exist_ok=True)
    nome = datetime.now().strftime("tempos_mds_%Y-%m-%d_%H-%M-%S")
    pasta = base / nome
    sufixo = 2
    while True:
        try:
            pasta.mkdir()
        except FileExistsError:
            pasta = base / f"{nome}_{sufixo:02d}"
            sufixo += 1
        else:
            return pasta


def carregar_csvs(pasta):
    pasta = Path(pasta).expanduser()
    execucoes = pd.read_csv(
        pasta / "mds_execucoes.csv",
        dtype={"run_id": str, "instancia": str},
        keep_default_na=False,
        na_values=[""],
        float_precision="round_trip",
    )
    caminho_historico = pasta / "mds_melhorias.csv"
    if not caminho_historico.is_file() and not execucoes["status"].eq("ok").any():
        historico = pd.DataFrame(columns=["run_id", "tempo_s", "cardinalidade"])
    else:
        historico = pd.read_csv(
            caminho_historico, dtype={"run_id": str}, float_precision="round_trip"
        )
    return execucoes, historico


def executar_instancia(G, instancia, seeds, limite_s, *, pasta=None):
    from src.complex_networks.algorithms.mds.demds import mds_solver_demds
    from src.complex_networks.algorithms.mds.greedy import mds_solver_greedy
    from src.complex_networks.algorithms.mds.iterated_greedy import mds_solver_iterated_greedy

    if not seeds or len(set(seeds)) != len(seeds):
        raise ValueError("Informe pelo menos uma semente, sem repetições.")

    algoritmos = (
        ("DemDS", mds_solver_demds, seeds, {"time_limit": ceil(limite_s)}),
        (
            "Iterated Greedy", mds_solver_iterated_greedy, seeds,
            {"time_limit": limite_s, "beta": IG_BETA},
        ),
        ("Guloso", mds_solver_greedy, seeds[:1], {"time_limit": limite_s}),
    )
    print(f"{instancia} | T = {limite_s:.3f} s", flush=True)
    resumos, historico = [], []
    for algoritmo, solver, sementes, argumentos in algoritmos:
        for seed in sementes:
            print(f"  {algoritmo} | seed {seed}", flush=True)
            resumo, eventos = medir_execucao(
                G, instancia, algoritmo, seed, limite_s, solver, argumentos
            )
            if resumo["status"] == "erro":
                print(f"    {resumo['erro']}", flush=True)
            if pasta is not None:
                salvar_linhas(pasta / "mds_melhorias.csv", eventos)
                salvar_linhas(pasta / "mds_execucoes.csv", [resumo])
            resumos.append(resumo)
            historico.extend(eventos)
    return pd.DataFrame(resumos), pd.DataFrame(
        historico, columns=["run_id", "tempo_s", "cardinalidade"]
    )

In [ ]:
def eventos_no_prazo(execucoes, historico):
    limites = execucoes.set_index("run_id")["limite_s"]
    return historico.loc[historico["tempo_s"].le(historico["run_id"].map(limites))]


def resumir_tempos_alvo(execucoes, eventos, otimo, gaps):
    colunas = [
        "instancia", "algoritmo", "gap_maximo_percentual", "cardinalidade_alvo",
        "n_execucoes", "n_execucoes_atingiram", "taxa_atingimento", "tempo_medio_ate_alvo_s",
    ]
    if otimo is None or execucoes.empty or not gaps:
        return pd.DataFrame(columns=colunas)

    resumos = []
    for gap in gaps:
        alvo = int(otimo * (1 + Fraction(str(gap)) / 100))
        tempos = eventos.loc[eventos["cardinalidade"].le(alvo)].groupby("run_id")["tempo_s"].min()
        por_seed = execucoes.assign(tempo_ate_alvo_s=execucoes["run_id"].map(tempos).astype(float))
        resumo = (
            por_seed.groupby(["instancia", "algoritmo"], sort=False)["tempo_ate_alvo_s"]
            .agg(n_execucoes="size", n_execucoes_atingiram="count", tempo_medio_ate_alvo_s="mean")
            .reset_index()
        )
        resumo["gap_maximo_percentual"] = gap
        resumo["cardinalidade_alvo"] = alvo
        resumo["taxa_atingimento"] = resumo["n_execucoes_atingiram"] / resumo["n_execucoes"]
        resumos.append(resumo[colunas])
    return pd.concat(resumos, ignore_index=True)

## Análise individual

Defina `INSTANCE_NAME` para rodar uma instância. Deixe
`PASTA_CSV_ANALISE = None` para executar os solvers ou informe a pasta
que contém `mds_execucoes.csv` e `mds_melhorias.csv` para usar um histórico
salvo. Para rodar somente a análise individual, pare antes de
**Relatórios em lote**.

A medição inclui a preparação interna do solver. A leitura do grafo,
a validação da solução e a gravação dos CSVs ocorrem fora desse intervalo.
O Guloso é determinístico e roda uma vez por grafo.

O DemDS recebe `time_limit` em segundos inteiros de CPU, arredondados
para cima. O Iterated Greedy e o Guloso recebem segundos decorridos.
O Iterated Greedy também para após 200 rodadas consecutivas sem melhora.

In [ ]:
INSTANCE_NAME = "p2p-Gnutella05"
SEEDS = (1,)
TIME_LIMIT_S = 20
SALVAR_CSV = True
PASTA_CSV_ANALISE = None
# PASTA_CSV_ANALISE = RESULTS_DIR / "benchmarks/mds/tempos_intermediarios/tempos_mds_..."

In [ ]:
if PASTA_CSV_ANALISE is None:
    from src.complex_networks.io import load_edgelist

    G = load_edgelist(RAW_DATA_DIR / f"{INSTANCE_NAME}.txt.gz")
    try:
        pasta_resultados = None
        if SALVAR_CSV:
            pasta_resultados = criar_pasta_resultados(
                RESULTS_DIR / "benchmarks" / "mds" / "tempos_intermediarios"
            )
            print(pasta_resultados)
        execucoes_analise, historico_analise = executar_instancia(
            G, INSTANCE_NAME, SEEDS, TIME_LIMIT_S, pasta=pasta_resultados
        )
    finally:
        del G
        gc.collect()
else:
    execucoes_analise, historico_analise = carregar_csvs(PASTA_CSV_ANALISE)

### Soluções e tempos até o alvo

A mediana em cada instante usa as sementes que já tinham uma solução
viável. A coluna `n_execucoes_com_solucao` informa quantas contribuíram.
A última solução registrada continua disponível depois que o solver termina.

Para os alvos, o notebook usa a coluna `dms` de `results/otimos.csv`.
O alvo de gap $g$ é $\lfloor\mathrm{dms}\,(1+g/100)\rfloor$. A média
do tempo é calculada entre as sementes que atingiram o alvo até o prazo.

In [ ]:
INSTANCIA_ANALISE = None  # Usa a primeira instância da execução ou do CSV.
SEED_ANALISE = None       # Inclui todas as sementes.
TEMPOS_CONSULTA_S = (1, 5, 10, 30, 60)
GAPS_PERCENTUAIS = (10, 5, 1, 0)

In [ ]:
def agregar_execucoes(selecionadas, historico, tempos_consulta, gaps, otimo):
    if selecionadas.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    eventos = (
        historico.loc[historico["run_id"].isin(selecionadas["run_id"])]
        .sort_values("tempo_s", kind="stable")
        .drop_duplicates(["run_id", "tempo_s"], keep="last")
    )
    curvas_por_run = {
        run_id: grupo.set_index("tempo_s")["cardinalidade"]
        for run_id, grupo in eventos.groupby("run_id", sort=False)
    }
    if not selecionadas["run_id"].isin(curvas_por_run).all():
        raise ValueError(
            "Há execuções válidas sem eventos. Confira se os dois CSVs são da mesma pasta."
        )

    tempo_maximo = selecionadas["tempo_total_s"].max()
    grade_consulta = pd.Index(
        [tempo_maximo, *tempos_consulta], dtype="float64", name="tempo_s"
    ).unique().sort_values()
    resumos, qualidades = [], []

    for (instancia, algoritmo), rodadas in selecionadas.groupby(
        ["instancia", "algoritmo"], sort=False
    ):
        identificacao = {"instancia": instancia, "algoritmo": algoritmo}
        n_execucoes = len(rodadas)
        curvas = [curvas_por_run[run_id] for run_id in rodadas["run_id"]]
        melhor = rodadas.sort_values(
            ["cardinalidade_final", "tempo_ate_melhor_s", "seed"]
        ).iloc[0]
        resumos.append({
            **identificacao,
            "n_seeds": rodadas["seed"].nunique(),
            "n_execucoes": n_execucoes,
            "tempo_total_medio_s": rodadas["tempo_total_s"].mean(),
            "tempo_medio_ate_melhor_s": rodadas["tempo_ate_melhor_s"].mean(),
            "cardinalidade_final_mediana": rodadas["cardinalidade_final"].median(),
            "melhor_cardinalidade": int(melhor["cardinalidade_final"]),
            "primeiro_tempo_melhor_s": melhor["tempo_ate_melhor_s"],
            "seed_melhor": melhor["seed"],
        })

        grade = grade_consulta
        for curva in curvas:
            grade = grade.union(curva.index)
        cardinalidades = pd.concat(
            [curva.reindex(grade, method="ffill") for curva in curvas],
            axis=1, ignore_index=True,
        )
        qualidades.append(pd.DataFrame({
            **identificacao,
            "tempo_s": grade,
            "cardinalidade_mediana": cardinalidades.median(axis=1).to_numpy(),
            "n_execucoes_com_solucao": cardinalidades.count(axis=1).to_numpy(),
            "n_execucoes": n_execucoes,
        }))


    tempos = resumir_tempos_alvo(
        selecionadas, eventos_no_prazo(selecionadas, eventos), otimo, gaps
    )
    return pd.DataFrame(resumos), pd.concat(qualidades, ignore_index=True), tempos


def plotar_qualidade(qualidade, resumo, tempo_maximo, otimo, seed):
    fig, ax = plt.subplots(figsize=(9, 4.5))
    dados_plot = qualidade.loc[qualidade["tempo_s"] <= tempo_maximo]
    anotacoes = []
    for rodada in resumo.itertuples(index=False):
        curva = dados_plot.loc[dados_plot["algoritmo"].eq(rodada.algoritmo)]
        linha, = ax.step(
            curva["tempo_s"], curva["cardinalidade_mediana"],
            where="post", label=rodada.algoritmo,
        )
        tempo = rodada.primeiro_tempo_melhor_s
        tamanho = rodada.melhor_cardinalidade
        cor = linha.get_color()
        ax.scatter(tempo, tamanho, color=cor, s=25, zorder=4)
        mediana = curva.set_index("tempo_s")["cardinalidade_mediana"].asof(tempo)
        anotacoes.append((mediana, tempo, tamanho, cor))

    # Escalone os rótulos por altura para evitar sobreposição.
    anotacoes.sort(key=lambda anotacao: anotacao[0])
    for indice, (mediana, tempo, tamanho, cor) in enumerate(anotacoes):
        esquerda = tempo > tempo_maximo / 2
        posicao_rotulo = offset_copy(
            ax.transData, fig=fig, x=-6 if esquerda else 6,
            y=8 + 16 * indice, units="points",
        )
        ax.annotate(
            f"Melhor {tamanho} em {tempo:.4g} s",
            xy=(tempo, tamanho),
            xytext=(tempo, mediana),
            textcoords=posicao_rotulo,
            ha="right" if esquerda else "left",
            va="bottom",
            color=cor,
            fontsize=9,
            arrowprops={"arrowstyle": "-", "color": cor, "lw": 0.8},
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.9},
        )
    if otimo is not None:
        ax.axhline(otimo, color="gray", linestyle="--", label="Ótimo de referência")
    ax.set(
        xlabel="Tempo decorrido (s)",
        ylabel="Mediana da melhor cardinalidade",
        xlim=(0, None),
    )
    escopo = "todas as sementes" if seed is None else f"seed {seed}"
    ax.set_title(f"{resumo['instancia'].iat[0]} | {escopo}")
    ax.margins(y=0.2)
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()

In [ ]:
instancia_analisada = INSTANCIA_ANALISE
if instancia_analisada is None and not execucoes_analise.empty:
    instancia_analisada = execucoes_analise["instancia"].iat[0]
rodadas = execucoes_analise.loc[execucoes_analise["instancia"].eq(instancia_analisada)]
if SEED_ANALISE is not None:
    rodadas = rodadas.loc[
        rodadas["seed"].eq(SEED_ANALISE) | rodadas["algoritmo"].eq("Guloso")
    ]

falhas = rodadas.loc[rodadas["status"].eq("erro"), ["algoritmo", "seed", "erro"]]
if not falhas.empty:
    display(falhas)
validas = rodadas.loc[rodadas["status"].eq("ok")]
otimo = otimos_mds.get(instancia_analisada)
resumo_agregado, qualidade_por_tempo, tempos_ate_alvo = agregar_execucoes(
    validas, historico_analise, TEMPOS_CONSULTA_S, GAPS_PERCENTUAIS, otimo
)

if resumo_agregado.empty:
    print("Nenhuma execução válida para a instância e a semente escolhidas.")
else:
    plotar_qualidade(
        qualidade_por_tempo, resumo_agregado, validas["tempo_total_s"].max(), otimo, SEED_ANALISE
    )
    display(resumo_agregado)
    consulta = qualidade_por_tempo.loc[
        qualidade_por_tempo["tempo_s"].isin(TEMPOS_CONSULTA_S)
    ].reset_index(drop=True)
    consulta["cardinalidade_mediana"] = consulta["cardinalidade_mediana"].astype(object).where(
        consulta["n_execucoes_com_solucao"].gt(0), "Sem solução até esse instante"
    )
    display(consulta)

    if otimo is None:
        print(f"{instancia_analisada}: sem referência em {RESULTS_DIR / 'otimos.csv'}.")
    elif not tempos_ate_alvo.empty:
        display(tempos_ate_alvo.assign(
            tempo_medio_ate_alvo_s=tempos_ate_alvo["tempo_medio_ate_alvo_s"]
            .astype(object).where(tempos_ate_alvo["n_execucoes_atingiram"].gt(0), "Alvo não atingido")
        ))

## Relatórios em lote

Execute as duas células desta seção depois de **Preparação**.
Use `SEEDS_LOTE` para configurar as sementes. O lote percorre todos
os `*.txt.gz` de `data/raw` e usa $T=\sqrt{0{,}1N}$ segundos por execução.
Os relatórios contam melhorias registradas até $T$ segundos decorridos.
O DemDS pode terminar depois de $T$ por usar um prazo interno de CPU.
Os dois CSVs são gravados diretamente em `RESULTS_DIR`.

- `mds_solucoes_T.csv` tem uma linha por instância e algoritmo. `sol`
  é a mediana das melhores cardinalidades até $T$ e `tempo` é a média
  dos tempos até elas, entre as sementes que encontraram uma solução.
  `melhor_sol` traz a menor cardinalidade observada. `melhor_algoritmo`
  identifica quem a encontrou, com desempate pelo menor tempo e depois
  pela ordem alfabética. As contagens mostram quantas sementes rodaram
  e quantas produziram solução até $T$.
- `mds_gaps.csv` tem uma linha por instância, com `gap50`, `gap25` e
  `gap10`. Vence a maior taxa de atingimento até $T$. Em empate, vence
  o menor tempo médio entre os atingimentos e depois a ordem alfabética.
  A célula fica vazia quando a instância está sem referência ou quando
  o alvo ficou sem atingimentos. Uma execução com erro conta como
  tentativa sem atingimento.

In [ ]:
def resumir_instancia_lote(execucoes, historico, otimo):
    validas = execucoes.loc[execucoes["status"].eq("ok")]
    eventos = eventos_no_prazo(validas, historico)
    melhores = eventos.sort_values(["cardinalidade", "tempo_s"]).drop_duplicates("run_id")
    por_seed = execucoes[["run_id", "instancia", "algoritmo", "limite_s"]].merge(
        melhores[["run_id", "cardinalidade", "tempo_s"]], on="run_id", how="left"
    )

    solucoes = por_seed.groupby(["instancia", "algoritmo", "limite_s"], sort=False).agg(
        sol=("cardinalidade", "median"),
        tempo=("tempo_s", "mean"),
        melhor_sol=("cardinalidade", "min"),
        n_seeds=("run_id", "size"),
        n_seeds_com_solucao=("cardinalidade", "count"),
    ).reset_index()
    solucoes["melhor_sol"] = solucoes["melhor_sol"].astype("Int64")
    ranking = por_seed.dropna(subset=["cardinalidade"]).sort_values(
        ["cardinalidade", "tempo_s", "algoritmo"]
    )
    solucoes["melhor_algoritmo"] = ranking["algoritmo"].iat[0] if not ranking.empty else ""

    gaps = (50, 25, 10, 5, 1)
    desempenho = resumir_tempos_alvo(execucoes, eventos, otimo, gaps)
    display(desempenho)
    vencedores = (
        desempenho.loc[desempenho["n_execucoes_atingiram"].gt(0)]
        .sort_values(
            ["taxa_atingimento", "tempo_medio_ate_alvo_s", "algoritmo"],
            ascending=[False, True, True],
        )
        .drop_duplicates("gap_maximo_percentual")
        .set_index("gap_maximo_percentual")["algoritmo"]
        .to_dict()
    )
    linha_gaps = {"instancia": execucoes["instancia"].iat[0]}
    linha_gaps.update({f"gap{gap}": vencedores.get(gap, "") for gap in gaps})
    return solucoes, linha_gaps, desempenho


def executar_lote_final(seeds):
    from src.complex_networks.io import load_edgelist
    from src.complex_networks.utils import iter_files

    # caminhos = list(iter_files(RAW_DATA_DIR, "*.txt.gz", order="size"))
    # if not caminhos:
    #     raise FileNotFoundError(f"Nenhum arquivo .txt.gz em {RAW_DATA_DIR}.")


    caminhos = sorted(
        list(iter_files(DATA_DIR / "raw", "*.txt.gz", order="size"))
        + list(iter_files(DATA_DIR / "extend", "*.txt.gz", order="size")),
        key=lambda caminho: caminho.stat().st_size,
    )

    if not caminhos:
        raise FileNotFoundError("Nenhum arquivo .txt.gz em raw ou extend.")
    solucoes_lote, gaps_lote, desempenhos_lote = [], [], []

    for caminho in caminhos:
        instancia = caminho.name.removesuffix(".txt.gz")
        G = load_edgelist(caminho)

        try:
            limite_s = sqrt(G.number_of_nodes() / 10)
            execucoes, historico = executar_instancia(
                G, instancia, seeds, limite_s
            )
        finally:
            del G
            gc.collect()

        otimo = otimos_mds.get(instancia)
        if otimo is None:
            print(f"  {instancia}: sem referência em otimos.csv.")

        solucoes, gaps, desempenho = resumir_instancia_lote(
            execucoes, historico, otimo
        )

        solucoes_lote.append(solucoes)
        gaps_lote.append(gaps)
        desempenhos_lote.append(desempenho)

    desempenho_lote = pd.concat(desempenhos_lote, ignore_index=True)

    # Uma linha por instância e gap, com os tempos médios dos algoritmos.
    tempos = desempenho_lote.pivot(
        index=["instancia", "gap_maximo_percentual"],
        columns="algoritmo",
        values="tempo_medio_ate_alvo_s",
    ).reindex(columns=["DemDS", "Iterated Greedy", "Guloso"])

    # Calcula as diferenças por instância e depois tira a média por gap.
    diferencas_tempos = pd.DataFrame({
        "DemDS-IG": tempos["DemDS"] - tempos["Iterated Greedy"],
        "DemDS-Greedy": tempos["DemDS"] - tempos["Guloso"],
        "IG-Greedy": tempos["Iterated Greedy"] - tempos["Guloso"],
    })

    diferencas_tempos = (
        diferencas_tempos.groupby(level="gap_maximo_percentual")
        .mean()
        .reindex([50, 25, 10, 5, 1])
        .rename_axis("gap")
        .reset_index()
    )

    # Escolhe o menor tempo médio entre os algoritmos em cada instância e gap.
    melhores_tempos = tempos.min(axis=1).unstack("gap_maximo_percentual")
    melhores_tempos = (
        melhores_tempos.reindex(
            index=[linha["instancia"] for linha in gaps_lote],
            columns=[50, 25, 10, 5, 1],
        )
        .rename(columns=lambda gap: f"gap{gap}")
        .rename_axis(columns=None)
        .reset_index()
    )

    return (
        pd.concat(solucoes_lote, ignore_index=True),
        pd.DataFrame(gaps_lote),
        diferencas_tempos,
        melhores_tempos,
    )

In [ ]:
SEEDS_LOTE = (1, 2, 3, 4, 5)

mds_solucoes_T, mds_gaps, diferencas_tempos, melhores_tempos = executar_lote_final(SEEDS_LOTE)
mds_solucoes_T.to_csv(RESULTS_DIR / "mds_solucoes_T_full.csv", index=False)
mds_gaps.to_csv(RESULTS_DIR / "mds_gaps_extend_full.csv", index=False)
diferencas_tempos.to_csv(RESULTS_DIR / "delta_t_full.csv", index=False)
melhores_tempos.to_csv(RESULTS_DIR / "melhores_tempos_full.csv", index=False)

display(mds_solucoes_T.astype(object).where(mds_solucoes_T.notna(), "—"))
display(mds_gaps)